# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [1]:
%%sh
git config --global user.email "ryzh.v.n@mail.ru"
git config --global user.name "isoniazid"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Initialized empty Git repository in /content/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


In [2]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Writing ml_pipeline.py


### Проверяем работоспособность пайплайна

In [3]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [4]:
%%writefile .gitlab-ci.yml
stages:
  - build
  - deploy

build_model:
  stage: build
  script:
    - echo "запускаем пайплайн..."
    - python ml_pipeline.py

  stage: deploy
  script:
    - echo "деплоим..."

Writing .gitlab-ci.yml


In [5]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log

[master (root-commit) 32f7cd3] build(ml_pipeline.py) добавлен пайплайн GitLab
 2 files changed, 26 insertions(+)
 create mode 100644 .gitlab-ci.yml
 create mode 100644 ml_pipeline.py
commit 32f7cd36810efa7d544217ffc400e3805324d14f (HEAD -> master)
Author: isoniazid <ryzh.v.n@mail.ru>
Date:   Sun May 3 18:02:35 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitLab


In [34]:
!git branch -M main
!git remote set-url origin /////////////Спрятал адрес


In [35]:
!git push -u origin main

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 790 bytes | 790.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/isoniazid/hw_mlops_ryzhkoff.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

```
https://github.com/isoniazid/hw_mlops_ryzhkoff/actions/runs/25287493565/job/74134154694
```

# NB NB NB NB NB NB NB NB NB NB NB NB NB NB NB
Поскольку:
- а) Гитлаб заблокан в РФ
- б) Мне не очень хотелось поднимать селфхост инстанс гитлаба
- в) И по условию задания было разрешено пользоваться другими гит-провайдерами (или, по крайней мере, иначе это задание было нереально выполнить)

Я сделал аналогичный пайплайн в гитхабе на github actions. За основу взял стандартный шаблон и чуть подкрутил его.

При этом .gitlab-ci я также приложил в репозиторий

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [37]:
!adr init doc/architecture/decisions
!adr new use_blue_green_strategy

/bin/bash: line 1: adr: command not found
/bin/bash: line 1: adr: command not found


# 2. use_blue_green_strategy

Date: 2026-04-04

## Status

Accepted

## Context

Выбирал какую стратегию деплоя выбрать из списка: Blue-Green, Canary, Rolling, Shadow

## Decision

Выбрал Blue-Green.

## Consequences

У нас стратегия, которая позволит мгновенно откатиться, если что не так, и при этом очень легко настраивается. Есть конечно проблема
с тем, что нужно вдвое больше ресурсов, но на нашем маленьком проекте эти требования в абсолютном значении скромные, так что можем себе позволить.

Почему не выбрал другие стратегиии:

### Canary
- По факту это "умный" blue-green, переход на который осуществляется не мгновенно, а по шагам. Для нашего маленького проекта нет смысл с этим заморачиваться, но потом, когда подрастем, можно будет переделать на Canary, тем более, что вся инфраструктура (балансировщик, инстансы) уже для этого будет.

### Rolling
- Вряд ли у нас больше двух инстансов на проде, поэтому rolling по-сути станет для нас тем же самым, что Canary на 50/50. В будущем в комплекте с Canary может быть неплохой стратегией, потому что будем экономно расходовать ресурсы, постепенно обновляться, и допускать до обновленных инстансов только часть трафика.

### Shadow
- У нас маленький проект, к которому легко написать пайплайны с тестами, или протестировать руками перед релизом. Если бы у нас была массивная инфраструктура, которую в случае ошибки долго переключать/откатывать, shadow имел бы смысл. А так у нас будет постоянно работающий в фоне инстанс, который отъедает ресурсы у пользователей, но не дает им никаких результатов. Зачем, если можно просто быстро переключиться на Green, и если результат нас не устроит, быстро откатиться на BLue?

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [38]:
%%writefile docker-compose.blue.yaml
version: '3.8'

services:
  app_blue:
    image: ml-service:v1.0.0
    container_name: app_blue
    ports:
      - "5001:5000"
    environment:
      - VERSION=v1.0.0

Writing docker-compose.blue.yaml


In [39]:
%%writefile docker-compose.green.yaml
version: '3.8'

services:
  app_green:
    image: ml-service:v1.1.0
    container_name: app_green
    ports:
      - "5002:5000"
    environment:
      - VERSION=v1.1.0


Writing docker-compose.green.yaml


In [40]:
%%writefile balancer.yaml
version: '3.8'

services:
  nginx:
    image: nginx:latest
    container_name: nginx_lb
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf
    depends_on:
      - app_blue
      - app_green

  app_blue:
    image: ml-service:v1.0.0

  app_green:
    image: ml-service:v1.1.0

Writing balancer.yaml


In [46]:
!ls

balancer.yaml		  docker-compose.green.yaml  requirements.txt
docker-compose.blue.yaml  ml_pipeline.py	     sample_data


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [47]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [48]:
!python ml_pipeline.py

Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [ ]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.x'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas
    - name: Run pipeline
      run: |
        python -c "import numpy as np; import pandas as pd; from sklearn.datasets import load_iris; from sklearn.model_selection import train_test_split; from sklearn.ensemble import RandomForestClassifier; from sklearn.metrics import accuracy_score; iris = load_iris(); X = iris.data; y = iris.target; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42); model = RandomForestClassifier(n_estimators=100, random_state=42); model.fit(X_train, y_train); y_pred = model.predict(X_test); accuracy = accuracy_score(y_test, y_pred); print(f'Accuracy: {accuracy:.2f}')"
    - name: Make pipeline reproducible
      run: |
        python -c "print('какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн?')"


Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [ ]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [ ]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master (root-commit) c29e765] build(ml_pipeline.py) добавлен пайплайн
 2 files changed, 35 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 ml_pipeline.py
commit c29e7659975b52391b4a07c33a329893f1f8426b (HEAD -> master)
Author: isoniazid <ryzh.v.n@mail.ru>
Date:   Sun Apr 3 19:13:08 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

```
https://github.com/isoniazid/hw_mlops_ryzhkoff/actions/runs/25291228668/job/74143233620
```

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.



- Очень понравился инструмент ADR, теперь хочу его на своей текущей работе использовать

- С пайплайнами для CI/CD до этого уже работал, так что ничего нового для себя не открыл (кроме того, что гитлаб, оказывается, в РФ давно заблокировали)))

- Считаю, что выбрал стратегию развертывания правильно, хотя почему-то за ее реализацию дается 1 балл, а за Canary 2, хотя она тут менее уместна под такой микропроект, на мой взгляд. В md-файликах в доках обосновал свой выбор.

- Про А\B тестирования понимаю, зачем оно нужно, но инфраструктуру разворачивать не стал.